# ta-seqscan Stop Detection

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

# Imports
import nomad.io.base as loader
import geopandas as gpd
from shapely.geometry import box
from nomad.stop_detection.viz import plot_stops_barcode, plot_time_barcode, plot_stops, plot_pings, _plot_base_geometry, plot_circles
import nomad.stop_detection.dbscan as DBSCAN
from nomad.stop_detection.density_based import seqscan_labels
import nomad.stop_detection.density_based as SEQSCAN


import nomad.data as data_folder
from pathlib import Path

from nomad.city_gen import City
import pandas as pd
import matplotlib.dates as mdates
import pandas as pd
import nomad.generation.viz as viz
from nomad.traj_gen import Agent, condense_destinations
import nomad.stop_detection.utils as utils

In [ ]:
# Parameters according to the config file
data_dir = Path(data_folder.__file__).parent
city = City.from_geopackage(data_dir / 'garden-city.gpkg')
buildings = gpd.read_parquet(data_dir / 'garden-city-buildings-mercator.parquet')

# Destination Diary for 2 stop trajectory

In [ ]:
start = '2024-06-01 00:00-08:00'

start_time = pd.date_range(start=start, periods=2, freq='90min')
unix_timestamp = [int(t.timestamp()) for t in start_time]
duration = [90]*2  # in minutes

location = ['w-x17-y10'] + ['r-x19-y11']


destinations = pd.DataFrame(
    {
        "datetime":start_time,
         "timestamp":unix_timestamp,
         "duration":duration,
         "location":location
    }
)

destinations = condense_destinations(destinations)
destinations

In [ ]:
seed = 50
dist_thresh = 12/15
time_thresh=120
dur_min=5
min_pts = 3
ha=15/15

Charlie = Agent(identifier="Andres",
                city=city)

Charlie.generate_trajectory(destination_diary=destinations,
                            seed=seed,
                            dt=0.15)

Charlie.sample_trajectory(beta_start=None,
                          beta_durations=None,
                          beta_ping=6,
                          seed=seed,
                          ha=ha)


### Plot normal stops

In [ ]:
traj = Charlie.sparse_traj
labels = seqscan_labels(
    traj,
    dist_thresh=10/15,
    min_pts=3
)
traj['cluster'] = labels

### Remove overlaps

In [ ]:
fig, (ax_map, ax_barcode) = plt.subplots(2, 1, figsize=(7,4.5),
                                         gridspec_kw={'height_ratios':[10,1]})

# Plot colored pings
plot_circles(traj, ax=ax_map, radius=0.1, color='cluster', cmap='PiYG', base_geometry=city.buildings_gdf, base_geom_color='#8c8c8c', base_geom_background='#d3d3d3')
plot_pings(traj, ax=ax_map, s=5, color='black')
plot_time_barcode(traj['timestamp'], ax=ax_barcode, set_xlim=True)
plot_time_barcode(traj, color='cluster', ax=ax_barcode, cmap='PiYG', set_xlim=False, lw=1.3)
ax_barcode.set_title("timestamps")
plt.tight_layout(pad=2)
plt.savefig('normal_clusters.svg' , format='svg')
plt.savefig('normal_clusters.png' , format='png', dpi=300)
plt.show()